# POC 05b — the dependence stack on every configuration

POC_05 ran the full estimator stack on one cell (`ec97004a74de`, commissioning `Lp0.1-s1`) and only
raw latent transfer gain on the others. This notebook runs **every estimator on every stored run,
individually**, and writes one bundle of long-form tables keyed by `run_key`, so that schedules,
losses and geometries can be compared on the same numbers.

* **Nothing trains.** Every run goes through `grid.assert_cached` before `analyse`, and a result
  that is not a cache hit is refused. A run the Store cannot read is recorded as a failed stage;
  the sweep moves on.
* **Nothing is re-estimated differently.** Every number comes from the same `traj` / `transfer`
  / `chain` / `partialdep` / `precond` calls POC_05 made. `fedwater_lab.sweep` adds only the loop,
  the per-stage error isolation, checkpointing and the bundle.
* **Resumable.** Each run is written to `parts/<run_key>/` as soon as it finishes; re-running the
  sweep cell skips finished runs. An interrupted sweep loses at most the run in progress.
* **The output to send back** is `POC_05_grid_bundle.zip` (a few hundred KB): `manifest.json` +
  `tables/*.parquet` + `worlds/*`. §5 prints its path.

## 0. Setup

Same environment rules as POC_05: `MKL_THREADING_LAYER=SEQUENTIAL` before NumPy (one OpenMP
runtime per process), torch's runtime started first. The loaded runtimes are **recorded in the
manifest**, not asserted: `torch\lib\libiompstubs5md.dll` is torch's stub library, not a second
runtime, so a prefix count would be a false alarm.

In [1]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"   # BEFORE numpy
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))

import sys, pathlib, json, time, threading, warnings
import numpy as np, pandas as pd

REPO = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "fedwater").is_dir()), None)
LAB_DIR = (REPO / "notebooks" / "fedwater_lab_refactor") if REPO else pathlib.Path.cwd()
for p in ([REPO / "src"] if REPO else []) + [LAB_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from fedwater_lab import grid as GD, sweep as SW
from fedwater_lab.checks import openmp_runtimes

try:
    import torch
    torch.get_num_threads()
    _ = torch.ones(256, 256) @ torch.ones(256, 256)
except ImportError:
    torch = None
_ = np.ones((256, 256)) @ np.ones((256, 256))
RUNTIMES = openmp_runtimes()
display(RUNTIMES[["user_api", "internal_api", "prefix", "threading_layer"]])
HAVE_REPO = REPO is not None
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("repo:", REPO, "| lab:", LAB_DIR)

,user_api,internal_api,prefix,threading_layer
0,blas,mkl,mkl_rt,sequential
1,openmp,openmp,libiomp,None
2,openmp,openmp,libiomp,None


repo: c:\Users\arthu\USPy\10_Mestrado\fedWater | lab: c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\fedwater_lab_refactor


## 1. Parameters

**Arms.** `pure` / `mixed` / `all` are the DECODED arm (class-level features of the decoded
prototypes, restricted to that slot class; needs the world and the repo). `latent` is the
RAW-LATENT arm (group-mean latent, PCA fitted on init; needs only the saved run).

**Which stage runs on which arm.** Transfer on all four; the chain, the partial correlation and
the onset test on `pure` and `latent` — they are the expensive stages and `pure` is the demand
claim. Change the tuples to widen it.

**Cost.** Measured on a saved run: the latent arm takes ~50 s at the defaults (transfer nulls
~28 s, chain nulls ~21 s). The decoded arm adds `analyse` (~7 s) and roughly 2.5 min of nulls.
Plan for **~3–3.5 min per run, ~2–2.5 h for 42 runs**; `QUICK=True` (30 draws) is about 1 h.

**p-value floor.** An empirical p over $n$ draws cannot go below $1/(n+1)$: 0.0099 at $n=100$,
0.032 at $n=30$. With 12 ordered pairs, a Bonferroni level of $0.05/12 = 0.0042$ needs
$n \ge 239$ — every table stores `p_floor` next to `p_emp` so this is never lost.

In [2]:
QUICK     = False            # True: 30 draws per null (~1 h instead of ~2-2.5 h)
OVERWRITE = False            # True: recompute runs already in parts/
SELECT    = None             # None = every run; or a list of run_keys

N = 30 if QUICK else 100
P = SW.SweepParams(
    block_months=3, bin_h=24, period_h=168, k_states=3,
    n_surr=N, n_rep=N, n_stab=10,
    transfer_arms=("pure", "mixed", "all", "latent"),
    chain_arms=("pure", "latent"), partial_arms=("pure", "latent"),
    onset_arms=("pure", "latent"),
    feats=("lvl", "amp", "p24"), n_comp=2,
    partial_feat_decoded="amp", partial_feat_latent="z0",
    p=1, min_train=4, lam=1e-2, control="mean", control_time="both", seed=0)

STORE_ROOT = LAB_DIR / "lab_sandbox"
RUNS_DIR   = STORE_ROOT / "runs"
OUT_DIR    = LAB_DIR / "poc05_grid" / ("quick" if QUICK else "full")
print(P)
print("out:", OUT_DIR)

SweepParams(block_months=3, bin_h=24, period_h=168, k_states=3, n_surr=100, n_rep=100, n_stab=10, transfer_arms=('pure', 'mixed', 'all', 'latent'), chain_arms=('pure', 'latent'), partial_arms=('pure', 'latent'), onset_arms=('pure', 'latent'), onset_top=3, onset_win=8, onset_smooth=3, feats=('lvl', 'amp', 'p24'), n_comp=2, partial_feat_decoded='amp', partial_feat_latent='z0', lams=(0.01, 0.05, 0.1, 0.2, 0.5), partial_lam=0.1, p=1, min_train=4, lam=0.01, control='mean', control_time='both', n_perm=19, seed=0)
out: c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\fedwater_lab_refactor\poc05_grid\full


## 2. What is stored, and what each number means

Every table is long-form and carries `run_key` (join to `configs` for the axes) and `arm`.

**Transfer** (`transfer`). Per ordered pair $c \to d$, fitted on blocks strictly before $b$ and
scored on $b$, calendar classes pooled as replicates, controls = the mean of the other clients at
$b$ and $b-1$ (both models):
$$G_{c\to d} = 1 - \frac{\mathrm{SSE}_{\text{full}}}{\mathrm{SSE}_{\text{restricted}}}, \qquad
r^2_{\text{restricted}} = 1 - \frac{\mathrm{SSE}_{\text{restricted}}}{\mathrm{SSE}_{\text{mean}}}.$$
Capacity null: $G$ recomputed with the source's blocks permuted (classes attached), $n$ draws;
`z` $=(G-\bar G_0)/s_0$, `p_emp` $=(1+\#\{G_0 \ge G\})/(1+n)$, `p_floor` $=1/(n+1)$.
`G_reversed` is $G$ on the time-reversed block axis. `r2_restricted` is the AR-R² diagnostic: if
$G$ tracks it across targets, an asymmetry is self-predictability.

**Onset** (`onset`, `rolling_gain`). $G(b)$ on a sliding window of 8 blocks, pooled over 3; for
the 3 pairs with the largest $|G_{c\to d}-G_{d\to c}|$, `in_ramp` says whether
$\arg\max_b G(b)$ lies in the transition, and `p_uniform` $= n_{\text{ramp}}/n_{\text{eval}}$ is
the exact p under a uniform peak.

**Chain** (`chain_gates`, `occupancy`, `occupancy_sim`, `stationarity`, `mi_te`, `mi_nulls`).
Codebook: $k$-means on init, shape features, frozen. `ari_mean` = ARI of half-sample refits
against the full fit; `distortion` $=1-\mathrm{SS}_{\text{within}}/\mathrm{SS}_{\text{total}}$.
Order test $G^2 = 2\sum_{ij} n_{ij}\log(\hat P_{ij}/\hat\pi_j)$, df $k(k-1)$; `dynamics` = order 0
rejected at 0.01. Occupancy $\pi_c(s)$ = share of blocks in state $s$; similarity
$1-\mathrm{JS}(\pi_a,\pi_b)/\log 2$. Stationarity: homogeneity $G^2$ between the two halves of a
phase. MI and $T_{c\to d}=I(s_d[b];\,s_c[b-1]\mid s_d[b-1])$ with Miller–Madow, plug-in biases
$(k-1)^2/2N$ and $k(k-1)^2/2N$. Nulls: `month blocks` permutes whole blocks of one client with its
classes attached (keeps the shared calendar); `fitted chains` simulates each client from its own
$\hat P$, starting at its observed first state (destroys the shared calendar). `degenerate` = all
draws equal, `p_emp` = NaN.

**Partial** (`partial_condition`, `partial`, `partial_structure`).
$\mathrm{pcorr}(c,d)=-\Theta_{cd}/\sqrt{\Theta_{cc}\Theta_{dd}}$, $\Theta=(R+\lambda I)^{-1}$,
over the penalty path. `estimable` = rows $\ge 4p$ and condition number $\le 30$. Structural AUC
$=P(|r_{\text{adjacent}}| > |r_{\text{other}}|)$ against the network's cross-district links —
ground truth, scoring only.

**Block-level reading** (`recoverability`, `ruler_stability`). $r$ over blocks between the
model-side feature and the client's own-demand feature; $r$ between the frozen-model trajectory
and the moving-ruler one (with its $n$ — for a commissioning run the history covers init only).

**Headline** (`headline`), one row per run × arm, with the axes attached. The one derived score:
$$\text{drift\_separation} = \overline{\Delta}_{\text{pairs} \ni \text{drift}} -
\overline{\Delta}_{\text{other pairs}}, \qquad \Delta = \mathrm{sim}_{\text{init}} -
\mathrm{sim}_{\text{final}},$$
positive when the drifted client's pairs lost more occupancy similarity than everyone else's.
It uses the drift district, so it is an evaluation score, never an input.

## 3. The index

Read through the Store the analysis will use. A run the Store cannot see (wrong directory
layout) would make `analyse` retrain; here it is flagged, and the sweep records its decoded
stage as failed instead of calling `analyse`.

In [3]:
store = load_world = None
if HAVE_REPO:
    from fedwater_lab.store import Store
    from fedwater_lab.worlds import load_world
    store = Store(STORE_ROOT)
    IDX = GD.scan(store.root / "runs")
    IDX["store_readable"] = [GD.locate(store, r.world_hash, r.run_key)["exists"]
                             for r in IDX.itertuples(index=False)]
else:
    IDX = GD.scan(RUNS_DIR)
    IDX["store_readable"] = False
IDX = GD.comparable_blocks(IDX)
if SELECT is not None:
    IDX = IDX[IDX["run_key"].isin(SELECT)].reset_index(drop=True)

print(f"{len(IDX)} runs | {IDX['cmp_block'].nunique()} comparability blocks | "
      f"{IDX['world_hash'].nunique()} worlds | store-readable: {int(IDX['store_readable'].sum())}")
if (~IDX["store_readable"]).any() and HAVE_REPO:
    display(IDX.loc[~IDX["store_readable"], ["run_key", "label", "dir"]])
display(IDX[["run_key", "label", "schedule", "proto_weight", "spec_term", "geometry",
             "rounds", "cmp_block"]])
display(GD.coverage(IDX))

42 runs | 7 comparability blocks | 4 worlds | store-readable: 42


,run_key,label,schedule,proto_weight,spec_term,geometry,rounds,cmp_block
0,2f3aba5dcac9,fmp__none+or__minmax_ref__a3w84s4__u30r712e5s0...,commissioning,0.0,0.0,a3w84s4,712,0
1,57aa26c2408f,fmp__none+or__minmax_ref__a3w84s4__u30r30e5s0_...,streaming,0.1,1.0,a3w84s4,30,0
2,694733f16841,fmp__none+or__minmax_ref__a3w84s4__u30r30e5s0_...,all,0.1,1.0,a3w84s4,30,0
3,8241f2e9ba64,fmp__none+or__minmax_ref__a3w84s4__u30r30e5s0_...,all,0.0,0.0,a3w84s4,30,0
4,d11b40a20d21,fmp__none+or__minmax_ref__a3w84s4__u30r30e5s0_...,streaming,0.0,0.0,a3w84s4,30,0
5,da72c9bbae27,fmp__none+or__minmax_ref__a3w84s4__u30r712e5s0...,commissioning,0.1,1.0,a3w84s4,712,0
6,1dced6f2762d,fmp~b02f4f__none__minmax_ref__a3w84s12__u30r5s0,all,NaN,NaN,a3w84s12,5,1
7,fe4d4a60301a,fpmp__none__minmax_ref__a3w84s12__u30r5s0,all,NaN,NaN,a3w84s12,5,2
8,028ca9bf44b3,fmp__none__minmax_ref__a3w84s12__u30r30e5s0__L...,all,0.1,0.0,a3w84s12,30,3
9,0f14e551356d,fmp__none__minmax_ref__a3w84s4__u30r30e5s0__Lp0,all,0.0,0.0,a3w84s4,30,4


,axis,levels,counts,min_cells,estimable
0,schedule,3,"{'all': 24, 'commissioning': 9, 'streaming': 9}",9,True
1,schedule_months,5,"{'None': 24, '0-11': 9, 'B6x129': 4, 'B6x24': ...",2,True
2,loss,6,"{'Lp0': 15, 'Lp0.1-s1': 13, 'upstream': 8, 'Ls...",1,False
3,proto_weight,4,"{'0.0': 15, '0.1': 15, 'nan': 8, '1.0': 4}",4,True
4,spec_term,3,"{'0.0': 18, '1.0': 16, 'nan': 8}",8,True
5,diff_term,3,"{'0.0': 32, 'nan': 8, '1.0': 2}",2,True
6,var_term,3,"{'0.0': 33, 'nan': 8, '1.0': 1}",1,False
7,scaling,1,{'minmax_ref': 42},42,False
8,transform,1,{'none': 42},42,False
9,orient,2,"{'True': 30, 'False': 12}",12,True


## 4. The sweep

One bar over runs; the clock keeps ticking inside a run (a background thread redraws it every
second), and the description names the run in progress. The first finished run prints a measured
estimate for the rest. Interrupting is safe: re-run this cell and finished runs are skipped.

In [4]:
from tqdm.auto import tqdm

bar = tqdm(total=len(IDX), desc="sweep", bar_format="{desc} {n}/{total} |{bar}| {elapsed}")
_stop = threading.Event()
def _tick():
    while not _stop.wait(1.0):
        bar.refresh()
threading.Thread(target=_tick, daemon=True).start()

_first = {}
def progress(i, n, label, secs):
    if i and "t" not in _first and secs > 0:
        _first["t"] = secs / i
        print(f"~{_first['t']:.0f} s per run -> about {_first['t'] * (n - i) / 60:.0f} min left")
    bar.n = i
    bar.set_description_str(f"sweep · {str(label)[-48:]}")
    bar.refresh()

t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    RESULT = SW.sweep(IDX, P, OUT_DIR, store=store, load_world=load_world,
                      decoded=HAVE_REPO, overwrite=OVERWRITE, progress=progress)
_stop.set(); bar.close()
print(f"sweep finished in {(time.time() - t0) / 60:.1f} min")
display(RESULT)

sweep 0/42 |          | 00:00

  File "c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Users\arthu\anaconda3\envs\TsLeaks\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Users\arthu\anaconda3\envs\TsLeaks\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\arthu\anaconda3\envs\TsLeaks\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


~227 s per run -> about 155 min left
sweep finished in 176.1 min


,run_key,seconds,n_failed_stages
0,2f3aba5dcac9,223.5,0
1,57aa26c2408f,218.5,0
2,694733f16841,217.3,0
3,8241f2e9ba64,216.6,0
4,d11b40a20d21,215.2,0
5,da72c9bbae27,224.0,0
6,1dced6f2762d,50.2,0
7,fe4d4a60301a,50.5,0
8,028ca9bf44b3,230.3,0
9,0f14e551356d,232.0,0


## 5. The bundle

`collect` concatenates the parts, writes `headline` and `manifest.json` (parameters, method hash,
package versions, the loaded OpenMP runtimes, table sizes, the data dictionary) and zips
`manifest.json` + `tables/` + `worlds/`. **Send that zip back.**

In [5]:
BUNDLE = SW.collect(OUT_DIR, IDX, P, extra={
    "openmp_runtimes": RUNTIMES[["user_api", "internal_api", "prefix", "threading_layer",
                                 "filepath"]].astype(str).to_dict("records"),
    "have_repo": HAVE_REPO})

cfg = BUNDLE["configs"]
print(f"runs done: {len(cfg)} | runs with a failed stage: {int((cfg['n_failed_stages'] > 0).sum())}")
err_cols = [c for c in cfg.columns if c.startswith("err:")]
if err_cols:
    display(cfg.loc[cfg[err_cols].notna().any(axis=1), ["run_key", "label"] + err_cols])
display(pd.Series(BUNDLE["manifest"]["tables"], name="rows").to_frame())
print("\nSEND THIS FILE:", BUNDLE["zip"], f"({BUNDLE['zip'].stat().st_size / 1024:.0f} KB)")

runs done: 42 | runs with a failed stage: 0


,run_key,label,err:decoded
6,1dced6f2762d,fmp~b02f4f__none__minmax_ref__a3w84s12__u30r5s0,no world / store / repo
7,fe4d4a60301a,fpmp__none__minmax_ref__a3w84s12__u30r5s0,no world / store / repo


,rows
chain_gates,82
mi_nulls,984
mi_te,492
occupancy,2784
occupancy_sim,1476
onset,246
partial,2460
partial_condition,82
partial_structure,160
recoverability,800



SEND THIS FILE: c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\fedwater_lab_refactor\poc05_grid\full\POC_05_grid_bundle.zip (378 KB)


## 6. A first look (no verdicts here)

The headline per run × arm, and two pivots over the axes. These are raw summaries for orientation;
the reading happens on the bundle.

In [6]:
H = BUNDLE["headline"]
show = [c for c in ("run_key", "arm", "schedule", "proto_weight", "spec_term", "geometry",
                    "pairs_clear_null", "G_max", "best_pair", "best_pair_p",
                    "codebook_stable", "dynamics", "mi_pairs_sig_monthblock",
                    "te_dirs_above_bias", "drift_separation", "partial_estimable",
                    "auc_partial_r") if c in H.columns]
display(H[show].sort_values(["arm", "schedule", "proto_weight"]).round(3))
for val in ("pairs_clear_null", "drift_separation"):
    if val in H.columns:
        display(H.pivot_table(index=["schedule", "proto_weight"], columns="arm",
                              values=val, aggfunc="mean").round(3))

,run_key,arm,schedule,proto_weight,spec_term,geometry,pairs_clear_null,G_max,best_pair,best_pair_p,codebook_stable,dynamics,mi_pairs_sig_monthblock,te_dirs_above_bias,drift_separation,partial_estimable,auc_partial_r
4,0f14e551356d,all,all,0.0,0.0,a3w84s4,0,0.087,District_D->District_A,0.842,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,274f83bbaa3a,all,all,0.0,0.0,a3w84s12,0,0.146,District_D->District_C,0.139,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,45eff5d7095d,all,all,0.0,0.0,a3w84s12,0,0.489,District_D->District_A,0.267,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,6a52b826462d,all,all,0.0,0.0,a3w84s4,0,0.066,District_B->District_D,0.079,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,8241f2e9ba64,all,all,0.0,0.0,a3w84s4,1,0.426,District_A->District_D,0.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,57aa26c2408f,pure,streaming,0.1,1.0,a3w84s4,0,0.024,District_C->District_D,0.406,True,True,3.0,5.0,0.017,True,0.125
92,7a43014bbc7d,pure,streaming,0.1,1.0,a3w84s4,1,0.190,District_C->District_B,0.030,True,True,0.0,6.0,0.015,True,0.111
128,bab14b26d627,pure,streaming,0.1,1.0,a3w84s4,0,0.091,District_D->District_B,0.317,True,True,3.0,6.0,0.285,True,0.444
132,bb03445c3043,pure,streaming,0.1,1.0,a3w84s4,1,0.372,District_B->District_D,0.010,False,True,0.0,4.0,0.693,True,0.556


arm                           all  latent  mixed   pure
schedule      proto_weight                             
all           0.0           0.571    1.00  0.857  0.571
              0.1           0.714    1.00  0.429  1.714
              1.0           0.500    1.50  0.000  0.000
commissioning 0.0           0.000    0.25  0.500  0.250
              0.1           1.250    0.25  0.750  0.750
              1.0           0.000    2.00  0.000  0.000
streaming     0.0           0.750    0.50  0.000  0.750
              0.1           0.000    0.25  0.250  0.500
              1.0           0.000    2.00  0.000  0.000

arm                         latent   pure
schedule      proto_weight               
all           0.0            0.055  0.035
              0.1            0.030  0.002
              1.0            0.002 -0.106
commissioning 0.0            0.000  0.203
              0.1            0.068  0.091
              1.0            0.133  0.190
streaming     0.0            0.074  0.093
              0.1            0.006  0.253
              1.0            0.442  0.627